# Clase 4 · 05 · Change Data Capture: una tabla vista como stream

Las bases operacionales (OLTP) **modifican** datos: actualizan y borran filas. ¿Cómo se entera el resto de los sistemas (un índice de búsqueda, una caché, el Lakehouse)? Escribir en todos desde la aplicación (*dual writes*) termina en inconsistencias.

**Change Data Capture (CDC)** extrae los cambios de la base como un **stream**: cada insert, update o delete es un evento. Herramientas como Debezium leen el log de MySQL o PostgreSQL y lo publican en Kafka.

En Delta esto se llama **Change Data Feed**: activando una propiedad de la tabla, cada versión registra qué filas cambiaron y cómo. Vamos a:

1. Crear una tabla de clientes con Change Data Feed.
2. Modificarla como lo haría una aplicación.
3. Leer sus cambios como stream.
4. Reconstruir la tabla **sólo a partir del log de cambios**.

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "02 - Escala")

In [ ]:
%run ../common/config

In [ ]:
%run ../common/generate_stream_data

In [ ]:
%run ../common/streaming_rules

In [ ]:
%run ../common/streaming_lab

In [ ]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

config = CourseConfig(spark.sql("SELECT current_catalog()").first()[0],
                      dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
spark.conf.set("spark.sql.session.timeZone", "UTC")
customers = qualified_table(config, "silver_customers")
products = qualified_table(config, "silver_products")
for table in [customers, products]:
    assert spark.catalog.tableExists(table), f"Falta {table}. Ejecutá la clase 2."
T = lambda name: qualified_table(config, name)

In [ ]:
LAB = "cdc"
reset_lab(spark, dbutils, config, LAB, ["stream_cdc_customers", "stream_cdc_log"])
source_table, log_table = T("stream_cdc_customers"), T("stream_cdc_log")

spark.sql(f"""
CREATE TABLE {source_table} USING DELTA
TBLPROPERTIES (delta.enableChangeDataFeed = true) AS
SELECT customer_id, country, segment, email FROM {customers} WHERE customer_id < 5
""")
display(spark.table(source_table).orderBy("customer_id"))

## 1. La aplicación modifica la tabla

Cinco operaciones, cada una genera una nueva versión de la tabla.

In [ ]:
operations = [
    ("cliente 1 pasa a premium", f"UPDATE {source_table} SET segment = 'premium' WHERE customer_id = 1"),
    ("cliente 2 se muda a CL", f"UPDATE {source_table} SET country = 'CL' WHERE customer_id = 2"),
    ("se elimina el cliente 3", f"DELETE FROM {source_table} WHERE customer_id = 3"),
    ("alta del cliente 100", f"INSERT INTO {source_table} VALUES (100, 'AR', 'new', 'nuevo@example.test')"),
    ("cliente 2 se muda a BR", f"UPDATE {source_table} SET country = 'BR' WHERE customer_id = 2"),
]
for description, statement in operations:
    spark.sql(statement)
    print("✔", description)
display(spark.sql(f"DESCRIBE HISTORY {source_table}").select("version", "operation").orderBy("version"))
display(spark.table(source_table).orderBy("customer_id"))

La tabla muestra sólo el **estado actual**: el cliente 2 está en BR, pero no sabemos que antes estuvo en CL, ni que existió el cliente 3.

## 2. Leer los cambios como stream

Con `readChangeFeed` leemos la tabla como un stream de **eventos de cambio**. Cada fila trae `_change_type` (`insert`, `update_preimage`, `update_postimage`, `delete`) y la versión en que ocurrió.

In [ ]:
changes = (spark.readStream.option("readChangeFeed", "true").option("startingVersion", 0)
           .table(source_table))
progress = run_stream(changes, lab_path(config, LAB, "checkpoints", "cdc"), table=log_table)
print(f"Eventos de cambio leídos: {input_rows(progress)}")
display(spark.table(log_table).orderBy("_commit_version", "customer_id", "_change_type"))

In [ ]:
by_version = (spark.table(log_table).groupBy("_commit_version", "_change_type").count()
              .toPandas().pivot(index="_commit_version", columns="_change_type", values="count").fillna(0))
colors = {"insert": "#16a34a", "update_preimage": "#cbd5e1", "update_postimage": "#2563eb", "delete": "#dc2626"}
fig, ax = plt.subplots(figsize=(9, 3.8))
bottom = np.zeros(len(by_version))
for change in [c for c in colors if c in by_version.columns]:
    ax.bar(by_version.index.astype(str), by_version[change], bottom=bottom, color=colors[change], label=change)
    bottom += by_version[change].to_numpy()
ax.set_xlabel("Versión de la tabla")
ax.set_ylabel("Eventos de cambio")
ax.set_title("El log de cambios: qué pasó en cada versión")
ax.legend()
plt.show()

Un `UPDATE` genera dos eventos: la fila **antes** (`update_preimage`) y **después** (`update_postimage`). Un `DELETE` también es un evento: en el log, borrar no hace desaparecer la historia.

## 3. Reconstruir la tabla desde el log

Si para cada `customer_id` nos quedamos con **el último cambio** (y descartamos los que terminan en `delete`), obtenemos el estado actual. Es la idea de **log compaction** en Kafka: guardar sólo el último valor por clave alcanza para reconstruir la base.

In [ ]:
from pyspark.sql.window import Window

last_change = Window.partitionBy("customer_id").orderBy(F.desc("_commit_version"))
rebuilt = (spark.table(log_table).where("_change_type != 'update_preimage'")
           .withColumn("rn", F.row_number().over(last_change)).where("rn = 1")
           .where("_change_type != 'delete'")
           .select("customer_id", "country", "segment", "email"))

current = spark.table(source_table).select("customer_id", "country", "segment", "email")
differences = rebuilt.exceptAll(current).count() + current.exceptAll(rebuilt).count()
print("¿La tabla reconstruida desde el log es igual a la original?", "Sí" if differences == 0 else f"No ({differences} diferencias)")
display(rebuilt.orderBy("customer_id"))

## 4. Lo que el log sabe y la tabla no

El historial del cliente 2 sólo existe en el log. Con el log se pueden construir **otras vistas** sin tocar la base original (la idea de **CQRS** y **event sourcing**).

In [ ]:
display(spark.table(log_table).where("customer_id = 2 AND _change_type IN ('insert', 'update_postimage')")
        .select("_commit_version", "customer_id", "country", "segment").orderBy("_commit_version"))

## Resumen

- **CDC** convierte los cambios de una base en un stream de eventos, evitando los *dual writes*.
- El estado (la tabla) y el stream (el log de cambios) son **dos caras** del mismo dato: del log se reconstruye el estado.
- Quedarse con el último valor por clave es la idea de **log compaction**.
- El log conserva información que el estado pierde (la mudanza intermedia a CL, el cliente borrado).